In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/merged_data.csv')

# Group by employee and calculate metrics
employee_metrics = df.groupby('employee_id').agg(
    total_tasks_assigned=('task_id', 'count'),
    tasks_completed=('completed', 'sum'),
    deadlines_met=('deadline_met', 'sum'),
    avg_efficiency=('efficiency_score', lambda x: x[x > 0].mean()),
    avg_satisfaction=('satisfaction_rating', lambda x: x[x > 0].mean()),
    avg_hours_estimated=('hours_estimated', 'mean'),
    avg_hours_actual=('hours_actual', lambda x: x[x > 0].mean()),
    high_priority_tasks=('priority', lambda x: (x == 'High').sum() + (x == 'Critical').sum()),
    department=('department_x', 'first'),
    role=('role', 'first'),
    years_experience=('years_experience', 'first')
).reset_index()

# Derived KPIs
employee_metrics['completion_rate'] = (
    employee_metrics['tasks_completed'] / employee_metrics['total_tasks_assigned'] * 100
).round(2)

employee_metrics['deadline_adherence_rate'] = (
    employee_metrics['deadlines_met'] / employee_metrics['tasks_completed'].replace(0, np.nan) * 100
).round(2)

employee_metrics['time_efficiency_ratio'] = (
    employee_metrics['avg_hours_estimated'] / employee_metrics['avg_hours_actual'].replace(0, np.nan)
).round(3)

# Composite Performance Score (weighted formula)
# This is the KEY metric you'll pitch to your manager
employee_metrics['performance_score'] = (
    (employee_metrics['completion_rate'] * 0.35) +
    (employee_metrics['deadline_adherence_rate'].fillna(0) * 0.30) +
    (employee_metrics['avg_efficiency'].fillna(0) * 0.20) +
    (employee_metrics['avg_satisfaction'].fillna(0) * 20 * 0.15)   # scale 1-5 to 0-100
).round(2)

# Performance tier classification
def classify_performance(score):
    if score >= 80: return 'High Performer'
    elif score >= 60: return 'Average Performer'
    elif score >= 40: return 'Needs Improvement'
    else: return 'At Risk'

employee_metrics['performance_tier'] = employee_metrics['performance_score'].apply(classify_performance)

print(employee_metrics.head())
employee_metrics.to_csv('../data/processed/employee_metrics.csv', index=False)

  employee_id  total_tasks_assigned  tasks_completed  deadlines_met  \
0     EMP0001                     9                9              4   
1     EMP0002                     9                9              3   
2     EMP0003                    10                8              1   
3     EMP0004                     7                7              3   
4     EMP0005                    13               11              5   

   avg_efficiency  avg_satisfaction  avg_hours_estimated  avg_hours_actual  \
0       75.511111          2.000000            29.666667         42.700000   
1       85.100000          3.333333            22.888889         36.622222   
2       78.150000          2.875000            20.600000         24.775000   
3       81.614286          2.428571            22.428571         31.785714   
4       79.263636          3.272727            23.923077         24.881818   

   high_priority_tasks department              role  years_experience  \
0                    4    Suppo

In [2]:
# Which task categories have the worst deadline adherence?
task_bottlenecks = df[df['completed'] == True].groupby('task_category').agg(
    total=('task_id', 'count'),
    on_time=('deadline_met', 'sum'),
    avg_overrun_hours=('hours_actual', 'mean')
).reset_index()

task_bottlenecks['on_time_rate'] = (task_bottlenecks['on_time'] / task_bottlenecks['total'] * 100).round(2)
task_bottlenecks_sorted = task_bottlenecks.sort_values('on_time_rate')
print(task_bottlenecks_sorted)

   task_category  total  on_time  avg_overrun_hours  on_time_rate
8       Training    204       71          28.307843         34.80
1    Client Call    213       76          24.841784         35.68
4    Feature Dev    199       73          27.681407         36.68
7        Testing    185       68          25.992432         36.76
5        Meeting    179       67          25.383799         37.43
2    Code Review    196       78          24.961224         39.80
6         Report    200       80          25.515500         40.00
3  Documentation    183       76          27.018579         41.53
0        Bug Fix    160       67          27.502500         41.88
